<i>École Polytechnique de Montréal</i>
<br>
<i>Cours PHS3104 : Mécanique quantique II</i>
<br>
<i>Session d'automne 2026</i>

<h3><center>Devoir 3 — Algorithme de Grover et coloriage de graphe</center></h3>

| Gabriel Poulin| Aiya Hamrouni | Nom Prénom |
|-------------------|-------------------|-------------------|
| 2379745 | 2364917| Matricule 0000000 |

In [7]:
%pip install qutip


     --------------------------------------- 10.8/10.8 MB 10.2 MB/s eta 0:00:00


**Initialisation du notebook**

In [17]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import Qobj, basis, tensor, qeye, expect
from qutip.measurement import measurement_statistics_povm
MAISON = {
    "couleurs": [("R", "V"), ("V", "B"), ("R", "B"),
                 ("V", "B"), ("R", "V")],
    "aretes": [(0, 1), (1, 2), (2, 3),
               (3, 0), (2, 4), (3, 4)]
}

TRIANGLE = {
    "couleurs": [("R", "V"), ("V", "B"), ("B", "R")],
    "aretes": [(0, 1), (1, 2), (2, 0)]
}

#### 4.1 Calculs analytiques [8 points]

**a) Phase et interférence [2 points]** — Pour le circuit de la figure 2a, avec $P(\phi) = \mathrm{diag}\left(1, e^{i\phi}\right)$, calculer l'état final et les probabilités de mesure après $P(\phi)H|0\rangle$, puis après $HP(\phi)H|0\rangle$. Montrer que la probabilité finale du résultat 1 vaut $\sin^2(\phi/2)$ et expliquer le rôle de la dernière porte de Hadamard.

**b) Action de la diffusion [2 points]** — Pour $|\psi\rangle = \sum_x a_x |x\rangle$, montrer que $D$ transforme chaque amplitude $a_x$ en $2\mu - a_x$, où $\mu = N^{-1}\sum_x a_x$.

**c) Une itération à la main [4 points]**  
**1.** Pour le triangle, identifier les chaînes valides et calculer les huit amplitudes après l'oracle, à partir de la superposition uniforme.

**2.** Calculer les amplitudes après une diffusion, donner le ket final et la probabilité totale de succès. Expliquer pourquoi les probabilités ne changent pas à l'étape de l'oracle.

#### 4.2 Construction des opérateurs avec QuTiP [15 points]

Vous devez créer les fonctions ci-dessous qui retourneront des `Qobj` sur les $n$ qubits, de taille $2^n \times 2^n$, avec `dims=[[2]*n, [2]*n]`.

**a) Projecteurs locaux [2 points]**  
**1.** Écrire `projecteur_local(n, k, b)`, qui retourne le projecteur $Q_k(b)$ défini en 3.2.

In [10]:
def projecteur_local(n,k,b):
    if k < 0 or k >= n:
        raise ValueError("Indice k invalide.")
    if b != 0 and b != 1: #b doit être 0 ou 1
        raise ValueError("b doit être égal à 0 ou 1.")
    facteurs = []
    for i in range(n):
        if i == k:
            facteurs.append(basis(2, b).proj())   
        else:
            facteurs.append(qeye(2))              
    return tensor(facteurs)

**2.** Pour $n = 3$, illustrer son action sur un état de base conservé et un état annulé.

In [18]:
n = 3
Q = projecteur_local(n, 1, 0) 

conserve= basis([2]*n, [0, 0, 1])     
annule= basis([2]*n, [0, 1, 1])  

print(Q.dims, Q.shape)
print((Q * conserve - conserve).norm())
print((Q * annule).norm())

#L'application du projecteur sur l'état conservé
# donne Q_1(0)|001⟩ = |001⟩. L'état est conservé 
# et la norme de la différence est de 0.0.
#Pour l'état annulé la norme de la projection est nulle. Donc
# la projection annule l'état si le bit k ne vaut pas b.

[[2, 2, 2], [2, 2, 2]] (8, 8)
0.0
0.0


**b) Contraintes et oracle [7 points]**

**1.** Écrire `conflit(n, u, v, couleurs)`, qui retourne $C_{uv}$ à partir des listes de couleurs. [2 points]

In [19]:
def conflit(n,u,v,couleurs):
    I = tensor([qeye(2) for _ in range(n)])
    C_uv = 0 * I
    for i in (0, 1):
        for j in (0, 1):

            if couleurs[u][i] == couleurs[v][j]:

                Q_u = projecteur_local(n, u, i)
                Q_v = projecteur_local(n, v, j)

                C_uv += Q_u * Q_v

    return C_uv


**2.** Donner le projecteur obtenu pour AB de la maison comme exemple de vérification. [1 point]

In [24]:
C = conflit(5, 0, 1, MAISON["couleurs"])
print(C) 
print((C - projecteur_local(5, 0, 1) * projecteur_local(5, 1, 0)).norm())
#En affichant le résultat de print((C - projecteur_local(5, 0, 1) * projecteur_local(5, 1, 0)).norm()),
# on calcule la norme de la différence entre cet opérateur de conflit et le
#  produit des deux projecteurs locaux. Le fait d'obtenir une norme de 0.0
#  confirme que l'opérateur de conflit correspond bien
#  à l'intersection des projecteurs de ces deux sous-systèmes.

Quantum object: dims=[[2, 2, 2, 2, 2], [2, 2, 2, 2, 2]], shape=(32, 32), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
0.0


**3.** Écrire `oracle(graphe)`, qui assemble $\Pi_{\text{val}}$ à partir des arêtes et retourne $O_f = I_N - 2\Pi_{\text{val}}$ sans programmer la liste des solutions (il devrait fonctionner pour la maison et le triangle). [2 points]

In [29]:

def oracle(graphe):
    couleurs = graphe["couleurs"]
    aretes = graphe["aretes"]
    n = len(couleurs)
    I = tensor([qeye(2) for i in range(n)])

    Pi_val = I

    for u, v in aretes:

        C_uv = conflit(n, u, v, couleurs)

        Pi_val = Pi_val * (I - C_uv)

    O_f = I - 2 * Pi_val

    return O_f
for nom, g in [("triangle", TRIANGLE), ("maison", MAISON)]:
    O = oracle(g)
    n = len(g["couleurs"])
    N = 2**n
    d = np.real(np.diag(O.full()))
    print(nom, O.dims, O.shape)
    print("  diagonal :", np.allclose(O.full(), np.diag(d)))
    print("  valeurs   :", np.unique(np.round(d, 10)))
    print("  nb de -1  :", int(np.sum(d < 0)))

triangle [[2, 2, 2], [2, 2, 2]] (8, 8)
  diagonal : True
  valeurs   : [-1.  1.]
  nb de -1  : 2
maison [[2, 2, 2, 2, 2], [2, 2, 2, 2, 2]] (32, 32)
  diagonal : True
  valeurs   : [-1.  1.]
  nb de -1  : 4


**4.** Justifier le fait que $O_f$ soit unitaire. Vérifier numériquement cette unitarité ainsi que le signe reçu par chaque état de base du triangle en utilisant une vérification indépendante des couleurs. [2 points]

In [ ]:
from itertools import product

def coloriage_valide(bits, graphe):
    couleurs = graphe["couleurs"]
    for u, v in graphe["aretes"]:
        if couleurs[u][bits[u]] == couleurs[v][bits[v]]:
            return False
    return True

def triangle_valide_bits(bits):
    a, b, c = bits
    conflit_AB = (a == 1 and b == 0) #Voir si il y a des bits ayant la même couleur
    conflit_BC = (b == 1 and c == 0)
    conflit_CA = (c == 1 and a == 0)
    return not (conflit_AB or conflit_BC or conflit_CA)

def verifier_oracle(graphe, test_valide, tol=1e-10):
    n = len(graphe["couleurs"])
    O_f = oracle(graphe)
    I = qeye([2] * n)

    # Unitarité
    erreur = (O_f.dag() * O_f - I).norm()
    print("Erreur d'unitarité :", erreur)
    assert erreur < tol

    # Signe de chaque état de base
    print("\nVérification des états de base :")
    for bits in product([0, 1], repeat=n):
        ket = basis([2] * n, list(bits))
        valide = test_valide(bits)
        signe_attendu = -1 if valide else 1
        erreur_etat = (O_f * ket - signe_attendu * ket).norm()
        print(bits, "| Valide :", valide, "| Signe :", signe_attendu,
              "| Erreur :", erreur_etat)
        assert erreur_etat < tol

verifier_oracle(TRIANGLE, triangle_valide_bits)   
verifier_oracle(MAISON, lambda bits: coloriage_valide(bits, MAISON))

Erreur d'unitarité : 0.0

Vérification des états de base :
(0, 0, 0) | Valide : True | Signe : -1 | Erreur : 0.0
(0, 0, 1) | Valide : False | Signe : 1 | Erreur : 0.0
(0, 1, 0) | Valide : False | Signe : 1 | Erreur : 0.0
(0, 1, 1) | Valide : False | Signe : 1 | Erreur : 0.0
(1, 0, 0) | Valide : False | Signe : 1 | Erreur : 0.0
(1, 0, 1) | Valide : False | Signe : 1 | Erreur : 0.0
(1, 1, 0) | Valide : False | Signe : 1 | Erreur : 0.0
(1, 1, 1) | Valide : True | Signe : -1 | Erreur : 0.0
Erreur d'unitarité : 0.0

Vérification des états de base :
(0, 0, 0, 0, 0) | Valide : False | Signe : 1 | Erreur : 0.0
(0, 0, 0, 0, 1) | Valide : False | Signe : 1 | Erreur : 0.0
(0, 0, 0, 1, 0) | Valide : False | Signe : 1 | Erreur : 0.0
(0, 0, 0, 1, 1) | Valide : True | Signe : -1 | Erreur : 0.0
(0, 0, 1, 0, 0) | Valide : True | Signe : -1 | Erreur : 0.0
(0, 0, 1, 0, 1) | Valide : False | Signe : 1 | Erreur : 0.0
(0, 0, 1, 1, 0) | Valide : False | Signe : 1 | Erreur : 0.0
(0, 0, 1, 1, 1) | Valide : Fal

In [ ]:
#Une matrice U est unitaire si U^dag * U = I. Ici, l'oracle s'écrit sous
#  la forme O_f = I_N - 2 * Pi_val, où Pi_val est un projecteur
#  (Pi^dag = Pi et Pi^2 = Pi), car chaque contrainte
#  est diagonale avec des 0 et des 1, et le produit de projecteurs qui 
# commutent reste un projecteur. Par conséquent :
# O_f^dag * O_f = (I_N - 2 * Pi_val)^2 
# = I_N - 4 * Pi_val + 4 * Pi_val^2 = I_N.
#Finalement, l'oracle inverse simplement certains signes 
# d'amplitudes sans modifier leurs modules, préservant ainsi les probabilités.

**c) Diffusion et circuit complet [6 points]**

**1.** Écrire `diffusion(n)` à partir de $H^{\otimes n}$ et du projecteur sur $|0^n\rangle$. [2 points]

In [42]:

def diffusion(n):
    H = Qobj(np.array([[1, 1],[1, -1]]) / np.sqrt(2))
    H_n = tensor([H for _ in range(n)])
    zero = tensor([basis(2, 0) for _ in range(n)])
    P0 = zero * zero.dag()

    I = tensor([qeye(2) for _ in range(n)])

    D = H_n * (2 * P0 - I) * H_n

    return D


**2.** Écrire `circuit(graphe, r)`, qui retourne $U_r = \left(DO_f\right)^r H^{\otimes n}$, sans mesure finale. [2 points]

In [41]:

def circuit(graphe, r):

    n = len(graphe["couleurs"])

    if r < 0 or int(r) != r:
        raise ValueError("r doit être un entier positif.")

    H = Qobj(
        np.array([[1, 1],[1, -1]]) / np.sqrt(2))
    H_n = tensor([H for _ in range(n)])

    O_f = oracle(graphe)

    D = diffusion(n)

    U_r = (D * O_f) ** int(r) * H_n

    return U_r


**3.** Pour le triangle, comparer l'état après une itération avec le résultat analytique. [2 points]

In [43]:

def resultat_analytique(graphe, test_valide):
    n = len(graphe["couleurs"])
    N = 2**n

    # Quelles chaînes sont valides ?
    valides = []
    for x in range(N):
        bits = [int(c) for c in format(x, f"0{n}b")]
        valides.append(test_valide(bits))
    M = sum(valides)

    # Amplitudes après oracle puis diffusion (départ : superposition uniforme)
    amp_valide   = (3*N - 4*M) / (N*np.sqrt(N))
    amp_invalide = (N - 4*M) / (N*np.sqrt(N))

    amplitudes = np.array([amp_valide if v else amp_invalide for v in valides])
    psi = Qobj(amplitudes.reshape(-1, 1), dims=[[2]*n, [1]*n])

    print("N =", N, "| M =", M)
    print("Amplitude valide   :", amp_valide)
    print("Amplitude invalide :", amp_invalide)
    return psi


graphe = TRIANGLE
n = len(graphe["couleurs"])

U = circuit(graphe, 1)
zero = basis([2]*n, [0]*n)

psi_numerique = U * zero
psi_theorique = resultat_analytique(graphe, triangle_valide_bits)

erreur = (psi_numerique - psi_theorique).norm()

print("Amplitudes numériques :", np.round(psi_numerique.full().ravel(), 6))
print("Amplitudes analytiques :", np.round(psi_theorique.full().ravel(), 6))
print("Erreur :", erreur)

N = 8 | M = 2
Amplitude valide   : 0.7071067811865475
Amplitude invalide : 0.0
Amplitudes numériques : [ 0.707107+0.j  0.      +0.j  0.      +0.j  0.      +0.j  0.      +0.j
  0.      +0.j -0.      +0.j  0.707107+0.j]
Amplitudes analytiques : [0.707107+0.j 0.      +0.j 0.      +0.j 0.      +0.j 0.      +0.j
 0.      +0.j 0.      +0.j 0.707107+0.j]
Erreur : 6.345765609839286e-16


#### 4.3 Étude numérique de l'amplification [17 points]

**a) Validation classique [2 points]** — Trouver les chaînes valides et les couleurs correspondantes à chaque sommet pour le graphe maison. Cette liste servira à valider l'oracle et les résultats.

**b) Interférences et amplification [15 points]**  
**1.** Pour les deux graphes et $r = 0, 1, 2, 3, 4$, calculer $|\psi_r\rangle$ et tracer la probabilité totale de succès (la somme des probabilités pour les chaînes valides) en fonction de $r$. Comparer à la formule de l'encadré et identifier le point d'arrêt près du premier maximum. [6 points]

**2.** Pour la maison, tracer l'histogramme des 32 probabilités au point d'arrêt choisi et identifier les chaînes valides sur ce graphique. [3 points]

**3.** Expliquer brièvement comment l'oracle puis la diffusion produisent l'amplification, pourquoi les probabilités ne changent pas immédiatement après l'oracle et pourquoi un plus grand nombre d'itérations n'est pas toujours préférable. [6 points]